# tbNFe - export em chunks

Notebook de operacao para baixar o relatorio `tbNFe - DIFAL NF-e por emitente`.

Fluxo:
1. carregar o helper local;
2. revisar os parametros;
3. rodar dry-run;
4. executar a exportacao com fallback mensal -> 15 dias -> 7 dias;
5. inspecionar `chunks`, `consolidado`, `top_n` e `manifest`.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "difal_report_chunks.py").exists():
            return candidate
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


ROOT = find_root()
WORKSPACE_ROOT = ROOT / "tb_nfe"
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import download_tb_nfe as mallha

ROOT, WORKSPACE_ROOT


In [ ]:
START_DATE = "2022-04-05"
END_DATE = "2025-12-31"
CHUNK_DAYS = None
PAGE_SIZE = 5000
TOP_N = 20
MAX_ATTEMPTS = 2
SLEEP_BETWEEN_ATTEMPTS = 5
OUTPUT_DIR = WORKSPACE_ROOT / "outputs"
DRY_RUN = False
RUN_EXPORT = True

{
    "START_DATE": START_DATE,
    "END_DATE": END_DATE,
    "CHUNK_DAYS": CHUNK_DAYS,
    "PAGE_SIZE": PAGE_SIZE,
    "TOP_N": TOP_N,
    "MAX_ATTEMPTS": MAX_ATTEMPTS,
    "OUTPUT_DIR": str(OUTPUT_DIR),
}


In [ ]:
plan = mallha.dry_run_plan(
    start_date=START_DATE,
    end_date=END_DATE,
    chunk_days=CHUNK_DAYS,
    page_size=PAGE_SIZE,
    top_n=TOP_N,
    output_dir=OUTPUT_DIR,
)
plan


In [ ]:
run = None
if RUN_EXPORT:
    run = mallha.run_tb_nfe_export(
        start_date=START_DATE,
        end_date=END_DATE,
        chunk_days=CHUNK_DAYS,
        page_size=PAGE_SIZE,
        top_n=TOP_N,
        output_dir=OUTPUT_DIR,
        max_attempts=MAX_ATTEMPTS,
        sleep_between_attempts=SLEEP_BETWEEN_ATTEMPTS,
        verbose=True,
    )
    run.summary
else:
    print("Set RUN_EXPORT = True to execute the export.")


In [ ]:
if run is not None:
    display(run.result.chunks.head())
    display(run.result.consolidated.head())
    display(run.result.top_n.head())
    display(run.result.manifest)
